# BRD4-BD1/BD2 joint-screen uncertainty audit

**Executed notebook.** Recomputes nearest BD1 training similarity for held-out molecules, evaluates domain-specific errors, and provides **illustrative** residual perturbations. No true biological selectivity probabilities or coverage guarantees are inferred.

## 1. Load the deposited model-ready and joint-screen datasets

In [1]:
import pandas as pd,numpy as np,zipfile
from rdkit import Chem,DataStructs,RDLogger
from rdkit.Chem import rdFingerprintGenerator
from pathlib import Path
RDLogger.DisableLog('rdApp.*')
root=Path('.'); joint=pd.read_csv('inputs/FDA_BD1_BD2_joint_predictions_1764.csv')
held1=pd.read_csv('inputs/BD1_heldout_predictions.csv'); modeling=pd.read_csv('inputs/BD1_model_ready_2816.csv')
held2=pd.read_csv('inputs/BD2_heldout_predictions.csv')


## 2. Compute held-out AD similarity and signed residuals

In [2]:
# Determine BD1 analog similarity of each heldout compound to scaffold-separated development.
heldSMI=set(held1.standardized_smiles)
train=modeling[~modeling.standardized_smiles.isin(heldSMI)]
assert len(train)==2275 and len(held1)==541
fpgen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=2048)
tr=[fpgen.GetFingerprint(Chem.MolFromSmiles(s)) for s in train.standardized_smiles]
test=[fpgen.GetFingerprint(Chem.MolFromSmiles(s)) for s in held1.standardized_smiles]
held1['similarity']=np.array([max(DataStructs.BulkTanimotoSimilarity(x,tr)) for x in test]);held2['similarity']=held2.nearest_training_tanimoto
held1['signed_error']=held1.predicted_pIC50-held1.observed_pIC50;held2['signed_error']=held2.predicted_pIC50-held2.observed_pIC50
held1['abs_error']=held1.signed_error.abs();held2['abs_error']=held2.signed_error.abs()


## 3. Residual perturbation sensitivity

The two residual samples are independently resampled as an explicitly imperfect diagnostic. Held-out records with similarity below 0.40 are sparse. The resulting percentiles are **not predictive confidence intervals**.

In [3]:
# High fidelity local estimates require enough heldout analogs; fallback to whole heldout distribution is explicit.
def local_errors(held,s):
    if s<.4:subset=held[held.similarity.between(0,.4,inclusive='left')]
    elif s<.6:subset=held[held.similarity.between(.4,.6,inclusive='left')]
    else:subset=held[held.similarity>=.6]
    return (subset, 'local' if len(subset)>=30 else 'global_fallback') if len(subset)>=30 else (held,'global_fallback')
rows=[];rng=np.random.default_rng(20261118)
for _,r in joint.loc[joint.both_AD03_unseen].iterrows():
    d1,s1=local_errors(held1,r.BD1_train_max_Tanimoto);d2,s2=local_errors(held2,r.BD2_train_max_Tanimoto)
    p1=d1.signed_error.to_numpy();p2=d2.signed_error.to_numpy();N=20000
    # Under an *illustrative* independent marginal residual exchangeability assumption only.
    delta=r.delta_pIC50_BD1_minus_BD2
    deltas=delta-rng.choice(p1,N)+rng.choice(p2,N)
    possible=np.quantile(deltas,[.05,.5,.95]);pBD2=float(np.mean(deltas<=-1));pBD1=float(np.mean(deltas>=1));pDual=float(np.mean((r.BD1_pred_pIC50-rng.choice(p1,N)>=6) & (r.BD2_pred_pIC50-rng.choice(p2,N)>=6)))
    rows.append(dict(drug_name=r.drug_name,BD1_pred_nM=r.BD1_pred_IC50_nM,BD2_pred_nM=r.BD2_pred_IC50_nM,BD1_AD=r.BD1_train_max_Tanimoto,BD2_AD=r.BD2_train_max_Tanimoto,predicted_delta_pIC50=delta,predicted_BD2_preference_fold=10**(-delta),BD1_error_reference=s1,BD1_error_pool_n=len(d1),BD2_error_reference=s2,BD2_error_pool_n=len(d2),BD1_reference_abs_error_p90=np.quantile(d1.abs_error,.9),BD2_reference_abs_error_p90=np.quantile(d2.abs_error,.9),exploratory_delta_5th=possible[0],exploratory_delta_median=possible[1],exploratory_delta_95th=possible[2],illustrative_P_BD2_preference_ge_10x=pBD2,illustrative_P_BD1_preference_ge_10x=pBD1,illustrative_P_both_pIC50_ge_6=pDual))
res=pd.DataFrame(rows)


## 4. Export the 29-drug audit and evaluate error by similarity

Compounds should not be called selective inhibitors without matched biochemical data.

In [4]:
# avoid misleading confidence intervals

res['qualitative_priority']=np.where((res.BD1_pred_nM<=1000)&(res.BD2_pred_nM<=1000),'putative_dual_potency',np.where((res.predicted_delta_pIC50<=-1)&(res.BD2_pred_nM<=1000),'BD2_pref_potent','other'))
res.sort_values(['qualitative_priority','BD2_pred_nM']).to_csv(root/'joint_screen_uncertainty_audit_29.csv',index=False)
held1[['standardized_smiles','observed_pIC50','predicted_pIC50','signed_error','abs_error','similarity']].to_csv(root/'BD1_heldout_local_AD_errors.csv',index=False)
summary=[]
for k,d in [('BD1',held1),('BD2',held2)]:
 for l,mask in [('lt_0.4',d.similarity<.4),('0.4_to_0.6',d.similarity.between(.4,.6,inclusive='left')),('ge_0.6',d.similarity>=.6),('overall',pd.Series(True,index=d.index))]:
  part=d[mask];summary.append(dict(domain=k,bin=l,n=len(part),MAE=part.abs_error.mean() if len(part) else np.nan,median_absolute_error=part.abs_error.median() if len(part) else np.nan,p90_absolute_error=part.abs_error.quantile(.9) if len(part) else np.nan))
pd.DataFrame(summary).to_csv(root/'heldout_error_by_AD_similarity.csv',index=False)
summary2={'n_FDA':len(joint),'n_both_AD03':int(joint.both_AD03_unseen.sum()),'n_both_AD04':int(joint.both_AD04_unseen.sum()),'n_both_AD05':int(joint.both_AD05.sum()),'n_both_under_1uM':int(((joint.BD1_pred_IC50_nM<=1000)&(joint.BD2_pred_IC50_nM<=1000)&joint.both_AD03_unseen).sum()),'n_BD2_10x':int((res.predicted_delta_pIC50<=-1).sum()),'n_BD1_10x':int((res.predicted_delta_pIC50>=1).sum())}
(root/'counts.txt').write_text(str(summary2));print(summary2);print(pd.read_csv(root/'heldout_error_by_AD_similarity.csv').to_string(index=False));print(res[['drug_name','predicted_delta_pIC50','illustrative_P_BD2_preference_ge_10x','exploratory_delta_5th','exploratory_delta_95th']].sort_values('illustrative_P_BD2_preference_ge_10x',ascending=False).head(10).to_string(index=False))


{'n_FDA': 1764, 'n_both_AD03': 29, 'n_both_AD04': 2, 'n_both_AD05': 0, 'n_both_under_1uM': 5, 'n_BD2_10x': 3, 'n_BD1_10x': 0}
domain        bin   n      MAE  median_absolute_error  p90_absolute_error
   BD1     lt_0.4  24 0.963678               0.767691            2.285080
   BD1 0.4_to_0.6  57 0.577407               0.508876            1.208056
   BD1     ge_0.6 460 0.378589               0.288697            0.817181
   BD1    overall 541 0.425493               0.315515            0.928715
   BD2     lt_0.4  12 1.586849               1.463421            2.329969
   BD2 0.4_to_0.6  21 0.751732               0.720843            1.140950
   BD2     ge_0.6 154 0.473666               0.412547            0.860875
   BD2    overall 187 0.576327               0.456805            1.117825
       drug_name  predicted_delta_pIC50  illustrative_P_BD2_preference_ge_10x  exploratory_delta_5th  exploratory_delta_95th
      retigabine              -1.176634                               0.52995      